# 01 · Chunking con LangChain en local

Este notebook trabaja en Python local y usa `LangChain` para inspeccionar una de las decisiones más importantes del sistema: cómo partimos los documentos.

## Objetivo
- Cargar documentos desde `docs/`
- Entender su metadato básico
- Crear chunks con `RecursiveCharacterTextSplitter`
- Comparar configuraciones
- Ver de forma tangible cómo afecta el chunking al RAG

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path("..") / ".env")
sys.path.insert(0, str((Path("..") / "app").resolve()))

from settings import Settings
from rag.chunking import build_splitter, chunk_documents, load_documents

settings = Settings.from_env()
docs_dir = Path("../docs")
documents = load_documents(docs_dir)
settings, len(documents)

## Paso 1 · Revisar documentos cargados

In [ ]:
[{"source": doc.metadata["source"], "doc_type": doc.metadata["doc_type"], "preview": doc.page_content[:180]} for doc in documents]

## Paso 2 · Construir el splitter de LangChain

In [ ]:
splitter = build_splitter(chunk_size=settings.chunk_size, overlap=settings.chunk_overlap)
splitter

## Paso 3 · Crear chunks indexables

In [ ]:
chunk_records = chunk_documents(documents, chunk_size=settings.chunk_size, overlap=settings.chunk_overlap)
len(chunk_records)

In [ ]:
chunk_records[:3]

## Paso 4 · Inspección manual

Busca señales de buen o mal chunking:
- cortes en mitad de una idea,
- chunks excesivamente largos,
- chunks demasiado cortos,
- pérdida de contexto entre fragmentos consecutivos.

In [ ]:
for chunk in chunk_records[:3]:
    print("=" * 100)
    print(f"Fuente: {chunk['source']} | chunk_id: {chunk['chunk_id']}")
    print(chunk["text"])
    print()

## Paso 5 · Comparar otra configuración

In [ ]:
alt_chunks = chunk_documents(documents, chunk_size=400, overlap=80)
{
    "config_actual": {"chunk_size": settings.chunk_size, "chunk_overlap": settings.chunk_overlap, "chunks": len(chunk_records)},
    "config_alternativa": {"chunk_size": 400, "chunk_overlap": 80, "chunks": len(alt_chunks)},
}

In [ ]:
alt_chunks[:2]

## Reflexión

Escribe una frase breve: ¿qué tipo de pregunta crees que empeora más cuando el chunking está mal hecho?

In [ ]:
reflexion = ""
reflexion